In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, utils, models, layers
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from ipywidgets import interact_manual

(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(x_train[i])
    label_idx = y_train[i][0]
    plt.title(f"{class_names[label_idx]}")
    plt.axis('off')
plt.show()

x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0
y_train_onehot = utils.to_categorical(y_train, 10)
y_test_onehot = utils.to_categorical(y_test, 10)

In [ ]:
model = models.Sequential([
    layers.Conv2D(16, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train_norm, y_train_onehot,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.2)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(x_test_norm, y_test_onehot, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

predictions = model.predict(x_test_norm)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = y_test.flatten()

plt.figure(figsize=(12, 5))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_test[i])
    pred_name = class_names[predicted_classes[i]]
    true_name = class_names[true_classes[i]]
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}")
    plt.axis('off')
plt.show()

In [ ]:
cm = confusion_matrix(true_classes, predicted_classes)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.show()

model.save('hw4-1.keras')

In [ ]:
def predict_image(index):
    plt.figure(figsize=(3, 3))
    plt.imshow(x_test[index])
    plt.axis('off')
    plt.show()
    print(f"Pred: {class_names[predicted_classes[index]]}")
    print(f"True: {class_names[true_classes[index]]}")

interact_manual(predict_image, index=(0, 9999))